# Chemical System Generation

The `t2fpharm.system.System` object holds all information about a chemical system,
such as a receptor or receptor–ligand complex.
It can be created from a PDB file using the `t2fpharm.system.from_pdb()` method.

In [1]:
import sciapi  # to fetch PDB files from the RCSB webserver
import t2fpharm

First, we need a PDB file. We can obtain one for example from the RCSB database using its PDB ID:

In [2]:
pdb_raw: str = sciapi.pdb.file.entry(pdb_id="1AQ1", file_format="pdb")

Depending on the source, you may want to fix the PDB file
to add missing hydrogens and other heavy atoms,
replace non-standard resiudes,
and remove unwanted chains:

In [3]:
(
    pdb_fixed, 
    added_residues,
    replaced_nonstandard_residues,
    added_heavy_atoms,
    added_terminals
) = t2fpharm.system.fix_pdb(
    file=pdb_raw, 
    remove_chain_ids=None, 
    add_missing_residues=True, 
    replace_nonstandard_residues=False, 
    add_missing_heavy_atoms=True, 
    add_missing_hydrogens=7.0,
    add_missing_hydrogens_forcefield=None, 
    keep_ids=True,
)

A `System` object can then be created from the PDB file:

In [4]:
sys: t2fpharm.system.System = t2fpharm.system.from_pdb(pdb_fixed)

## Methods and Attributes

The `System` object provides various methods and attributes to analyze, visualize, and modify the the structure.
A small subset of these are demonstrated below.

Visualizing the structure:

In [5]:
sys.display()

ThemeManager()

NGLWidget(gui_style='ngl')

Accessing the composition (i.e., atomic data corresponding to the ATOM records of the PDB file) as a `pandas.DataFrame` object:

In [6]:
sys.composition.atoms

,chain_id,res_name,res_seq,i_code,res_poly,res_std,serial,name,alt_loc,occupancy,temp_factor,element,charge,element_index
serial,,,,,,,,,,,,,,
1,A,MET,1,,True,True,1,N,,1,0,N,<NA>,6
2,A,MET,1,,True,True,2,H,,1,0,H,<NA>,0
3,A,MET,1,,True,True,3,H2,,1,0,H,<NA>,0
4,A,MET,1,,True,True,4,H3,,1,0,H,<NA>,0
5,A,MET,1,,True,True,5,CA,,1,0,C,<NA>,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5281,A,HOH,423,,False,False,5281,H1,,1,0,H,<NA>,0
5282,A,HOH,423,,False,False,5282,H2,,1,0,H,<NA>,0
5283,A,HOH,424,,False,False,5283,O,,1,0,O,<NA>,7


Accessing the trajectory data as a `jax.Array` object:

In [7]:
sys.trajectory.points

Array([[-14.259,  32.186,  -3.871],
       [-14.54 ,  33.095,  -3.563],
       [-14.411,  33.161,  -3.71 ],
       ...,
       [ 21.594,  25.739,  42.908],
       [ 22.263,  26.44 ,  43.156],
       [ 21.961,  26.626,  43.189]], dtype=float32)

The trajectory and atoms lists have the same length and order:

In [8]:
len(sys.trajectory.points) == len(sys.composition.atoms)

True

We can select a subset of the system by providing a mask into `system.composition.atoms`; for example to select the polymeric entities:

In [9]:
receptor: t2fpharm.system.System = sys.select(sys.composition.atoms["res_poly"])
receptor.display()

NGLWidget(gui_style='ngl')

We can convert the system to a PDBQT file:

In [10]:
pdbqt: str = sys.to_pdbqt(
    autobond=False,
    rigid=True,
    combine=False,
    flexible=False,
    preserve_serials=True,
    preserve_hydrogens=False,
    preserve_names=True,
    charge_model="gasteiger",
    add_hydrogens=False,
    protonation_ph=None,
)
# Print the first 10 lines
print("\n".join(pdbqt.splitlines()[:10]))

REMARK  Name = 
REMARK                            x       y       z     vdW  Elec       q    Type
REMARK                         _______ _______ _______ _____ _____    ______ ____
ATOM      1  N   MET A   1     -14.259  32.186  -3.871  0.00  0.00    -0.163 NA
ATOM      2  H   MET A   1     -14.540  33.095  -3.563  0.00  0.00    +0.000 HD
ATOM      3  H2  MET A   1     -14.411  33.161  -3.710  0.00  0.00    +0.163 HD
ATOM      4  H3  MET A   1     -14.585  33.114  -3.690  0.00  0.00    +0.000 HD
ATOM      5  CA  MET A   1     -13.528  30.893  -3.696  0.00  0.00    +0.100 C 
ATOM      6  C   MET A   1     -13.109  30.319  -5.039  0.00  0.00    +0.218 C 
ATOM      7  O   MET A   1     -12.211  29.492  -5.150  0.00  0.00    -0.275 OA
